# **OFDM** 

### ¿Qué es?
OFDM (Multiplexación por División de Frecuencia Ortogonal) es un método de modulación multiportadora digital para transmisión de símbolos. Divide un canal de alta velocidad en múltiples subportadoras de baja velocidad. Estas subportadoras están matemáticamente diseñadas para ser ortogonales, lo que permite que sus espectros se solapen sin interferirse mutuamente, transmitiendo datos en paralelo, mejorando la eficiencia espectral. 

Básicamente, en lugar de enviar los datos en una sola portadora, OFDM divide la señal en muchas señales más pequeñas y las transmite simultáneamente, permitiendo mayor resistencia a la interferencia y a la dispersión en el canal.

Uno de los elementos clave de la implementación de OFDM es el uso de la transformada discreta de Fourier (DFT) y su inversa (IDFT), que permiten una conversión eficaz de la señal entre los dominios de la frecuencia y el tiempo.

### ¿En qué se usa OFDM?
Se utiliza en los sistemas modernos de comunicaciones digitales de banda ancha, especialmente en entornos con multipropagación. Por ejemplo en comunicaciones WiFi, Redes celulares 4G y 5G, Televisión digital terrestre, etc.

---

# Transmisor OFDM

La información que se quiere transmitir es una secuencia de bits.
La agrupación de estos bits forman símbolos, que luego son modulados digitalmente, por ejemplo en modulación **QPSK, QAM16, QAM64, etc**.
Al usar estas modulaciones los símbolos quedan mapeados como números complejos, obteniendo así una secuencia de **símbolos complejos**.

## Símbolos Complejos
Los símbolos complejos resultantes se asignan a las distintas subportadoras en el dominio de la frecuencia. La **IDFT** permite transformar esta representación en frecuencia al dominio temporal para su transmisión física. 

## Señal OFDM en tiempo discreto
Para transmitir los símbolos en paralelo sobre múltiples subportadoras ortogonales, la IDFT genera en el dominio temporal la combinación lineal de exponenciales complejas ortogonales.
Para implementarlo, se generan $N$ muestras discretas que constituyen un símbolo OFDM en el dominio temporal.
Podemos definir $T$ como el periodo de cada símbolo OFDM y $T_s$ como el periodo de muestreo $(T_s=T/N)$.
Con un paso de $t=nT_s$ ($t=nT/N$), la señal OFDM se determina como:

$$
x[n] = \frac{1}{N} \sum_{k=0}^{N-1} X[k] \, e^{j 2\pi \frac{kn}{N}}
$$

Para $n=0,1,2,...,N-1$

Donde:
- $X[k]$ son los símbolos complejos asignados a cada subportadora.
- Cada término exponencial representa una subportadora ortogonal.
- $\frac{1}{N}$ es el factor de normalización de la IDFT, necesario para que la DFT aplicada en el receptor permita recuperar los símbolos originales $X[k]$. 
- Cada muestra $x[n]$ es una combinación de todas las subportadoras.
- El bloque completo de $N$ muestras forma un **símbolo OFDM**

## Garantía de Ortogonalidad
Si definimos dos vectores complejos $\phi_k$ y $\phi_m$, la ortogonalidad se da cuando al calcular el producto interno entre ellos es igual a 0,  $\langle \phi_k , \phi_m\rangle = 0$.

Por lo tanto cada subportadora es ortogonal entre sí, si se cumple que para:

$$
\langle \phi_k[n] , \phi_m[n]\rangle = \sum_{n=0}^{N-1} e^{j 2\pi \frac{kn}{N}}.e^{-j 2\pi \frac{mn}{N}}= \sum_{n=0}^{N-1}e^{j 2\pi (k-m)\frac{n}{N}} = \ 0 \ para \ k \neq m.
$$

Si definimos $r=e^{j 2\pi \frac{(k-m)}{N}}$, y reemplazamos en la sumatoria, obtenemos una serie geométrica compleja $S$.
$$
S = \sum_{n=0}^{N-1}r^n = \frac{1-r^N }{1-r} \ si \ r \neq 1
$$
Para cumplir con la condición de que $\langle \phi_k[n] , \phi_m[n]\rangle = 0$, $\ r^N$ debe ser 1.
$$
r^N = (e^{j 2\pi \frac{(k-m)}{N}})^N = e^{j 2\pi(k-m)}
$$
Como $k$ y $m$ son números enteros $\to \ e^{j 2\pi(k-m)} = 1$.

La ortogonalidad entre subportadoras se garantiza cuando el espaciamiento en frecuencia cumple $\Delta f=1/T$.

---
## Prefijo Cíclico  
En un canal con múltiple trayecto, la señal transmitida sufre retardos debido a reflexiones.
Esto produce interferencia entre símbolos OFDM consecutivos (ISI) y puede romper la ortogonalidad entre subportadoras.
Para evitar este problema, se inserta un prefijo cíclico (CP), que consiste en copiar las últimas muestras del símbolo OFDM y añadirlos al inicio del mismo.

#### ¿Cómo determino la cantidad de muestras para el $CP$?
La longitud del CP debe compensar el máximo retardo entre el camino directo y el camino más tardío.
Esto es la **dispersión del canal**.
En el dominio del tiempo la señal recibida es la convolución del símbolo transmitido y la respuesta al impulso del canal.
$$
r[n] = x[n] \circledast h[n]
$$

Si la respuesta al impulso del canal $h[n]$ tiene una longitud de $L_h$ muestras, el máximo retardo introducido por el canal es $(L_h-1)$ muestras. Por lo tanto, se debe cumplir que la longitud de $CP$ tiene que ser:
$$
L_{CP} \geq L_h-1
$$ 
Para no romper la ortogonalidad de las subportadoras y evitar ISI.

Para señales como WiFi, LTE, Televisión digital, el $CP$ está estandarizado.

---
## Transmisión por antena
Una vez obtenido el símbolo OFDM $x[n]$, la señal en tiempo discreto pasa por un conversor digital-analógico (DAC) para transformarse en una señal analógica continua $x(t)$.

Como $x[n]$ es una banda base compleja discreta, $x(t)$ resulta una banda base compleja continua, y se modula a una frecuencia de portadora $f_c$ de alta frecuencia, produciendo un corrimiento espectral, para luego transmitir por antena.

#### Señal Resultante
Si $x(t) = I(t)+jQ(t)$, la portadora es $e^{j 2\pi f_c t}$, la señal resultante es: 
$
s(t)= \real \{ x(t).e^{j 2\pi f_c t} \}
$

Reemplazando: 
$$
s(t)= I(t)cos(2\pi f_c t)-Q(t)sen(2\pi f_c t)
$$

---

### Imagen ilustrativa:

![transmisor_ideal](../img/imagen-transmisor.png)

---


# Receptor OFDM

El objetivo es recuperar la secuencia de bits transmitida a partir de la señal OFDM recibida.

En un sistema real, la señal analógica recibida por la antena se demodula a una frecuencia de portadora $f_c$ y pasa por un conversor analógico-digital (ADC), que realiza el muestreo y cuantización de la señal para obtener una representación digital de la misma. Esta señal digital es la que posteriormente puede ser procesada por el receptor OFDM.

---

## Señal recibida

Se considera una transmisión en **banda base** y un **canal ideal**, por lo tanto no hay ruido, desvanecimiento, interferencia ni distorsión del canal. Por lo tanto, podemos considerar:

$$
r[n] = x[n]
$$

donde:

* $x[n]$ es la señal OFDM transmitida.
* $r[n]$ es la señal OFDM recibida.
* $n=0,1,...,N-1$ representa las muestras del símbolo OFDM.

---

## Eliminación del Prefijo Cíclico

Antes de aplicar la DFT es necesario eliminar el prefijo cíclico agregado en el transmisor.

Si el símbolo OFDM tiene $N$ muestras y el prefijo cíclico tiene una longitud $L_{CP}$, la señal recibida tendrá $N+L_{CP}$ muestras.

El receptor elimina las primeras $L_{CP}$ muestras, conservando únicamente las $N$ muestras correspondientes al símbolo OFDM original.

A partir de este punto, $r[n]$ representa las $N$ muestras del símbolo OFDM recibidas una vez eliminado el prefijo cíclico.

---

## Transformación al dominio de la frecuencia

Ya con las $N$ muestras del símbolo OFDM disponibles, para recuperar los símbolos complejos que fueron asignados a las subportadoras, se aplica la DFT.

La DFT de la señal recibida se define como:

$$
Y[k] = \sum_{n=0}^{N-1} r[n] e^{-j2\pi\frac{kn}{N}}
$$

Para: $k=0,1,...,N-1$

La DFT permite transformar nuevamente la representación de la señal desde el dominio temporal hacia el dominio de la frecuencia.

Debido a que es un canal ideal, y considerando que la DFT y la IDFT son operaciones inversas entre sí, se obtiene:

$$
Y[k] = X[k]
$$

Por lo tanto, los símbolos complejos recuperados en cada subportadora son exactamente iguales a los símbolos transmitidos.

---

## Demodulación

Los símbolos complejos recuperados mediante la DFT deben ser demodulados para obtener nuevamente la secuencia de bits original.

La demodulación realiza el proceso inverso al mapeo realizado en el transmisor.

En QPSK los símbolos recibidos se encuentran en cuatro posibles posiciones del plano complejo:

$$
X[k] \in
\left\{
1+j, -1+j, -1-j, 1-j
\right\}
$$

Cada posición representa una combinación diferente de dos bits.

El demodulador determina a qué símbolo de la constelación pertenece cada valor complejo recibido y asigna nuevamente la combinación de bits correspondiente.

Finalmente, los grupos de bits obtenidos de cada símbolo se concatenan para formar nuevamente la secuencia de bits transmitida.

---

### Imagen ilustrativa:

![receptor_ideal](../img/imagen-receptor.png)

---


## Programa en Python:

In [ ]:
# Librerías
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8,5)
plt.rcParams["font.size"] = 12

np.random.seed(10)   # Para que siempre genere los mismos bits

In [ ]:
# Configuración de parámetros de la simulación

N = 64          # Número de subportadoras OFDM
CP = 16         # Longitud del prefijo cíclico
N_OFDM = 500    # Cantidad de símbolos OFDM

k = 2           # QPSK: 2 bits por símbolo

# Cantidad total de bits
N_BITS = N * N_OFDM * k

In [ ]:
# Generación de bits

bits_tx = np.random.randint(0, 2, N_BITS)

print("Bits generados:", len(bits_tx))
print("Primeros 10 bits generados:", bits_tx[:10])

In [ ]:
# Modulador QPSK
def qpsk_modulator(bits):
    """
    Convierte una secuencia de bits en símbolos QPSK complejos.

    Cada símbolo QPSK representa 2 bits.

    Mapeo:

        00 ->  1 + 1j
        01 -> -1 + 1j
        11 -> -1 - 1j
        10 ->  1 - 1j

    Los símbolos se normalizan para tener potencia promedio unitaria.
    """

    bits = bits.reshape((-1, 2)) # Agrupa los bits de 2 en 2

    constellation = {
        (0, 0):  1 + 1j,
        (0, 1): -1 + 1j,
        (1, 1): -1 - 1j,
        (1, 0):  1 - 1j
    }

    symbols = []

    for b in bits:
        symbol = constellation[tuple(b)]
        symbols.append(symbol)

    symbols = np.array(symbols)

    # Normalización
    return symbols / np.sqrt(2)

In [ ]:
# Demodulador QPSK
def qpsk_demodulator(symbols):
    """
    Convierte símbolos QPSK complejos nuevamente en bits.

    La decisión se realiza según el cuadrante
    en el que se encuentra cada símbolo.
    """

    # Deshacer la normalización
    symbols = symbols * np.sqrt(2)

    bits = []

    for s in symbols:

        I = s.real
        Q = s.imag

        if I >= 0 and Q >= 0:
            bits.extend([0, 0])

        elif I < 0 and Q >= 0:
            bits.extend([0, 1])

        elif I < 0 and Q < 0:
            bits.extend([1, 1])

        else: # I >= 0 and Q < 0
            bits.extend([1, 0])

    return np.array(bits)

In [ ]:

# FUNCIONES OFDM
def serial_to_parallel(symbols, N):
    """
    Convierte un vector de símbolos en una matriz.
    Cada fila representa un símbolo OFDM.
    Cada columna representa una subportadora.
    """
    return symbols.reshape((-1, N))

def parallel_to_serial(matrix):
    """
    Convierte una matriz nuevamente en un vector.
    """
    return matrix.reshape(-1)

def add_cp(ofdm_symbol, CP):
    """
    Agrega el prefijo cíclico.
    """
    cp = ofdm_symbol[-CP:]
    return np.concatenate((cp, ofdm_symbol))

def remove_cp(ofdm_symbol, CP):
    """
    Elimina el prefijo cíclico.
    """
    return ofdm_symbol[CP:]

In [ ]:
# TRANSMISOR OFDM

def ofdm_transmitter(symbols, N, CP):
    """
    Transmisor OFDM.
    Etapas:
        Serie → Paralelo
        IFFT
        Agregar CP
        Paralelo → Serie
    """
    # Serie -> Paralelo
    symbols_matrix = serial_to_parallel(symbols, N)
    tx_signal = []

    # Procesar cada símbolo OFDM
    for block in symbols_matrix:

        # Transformación al dominio del tiempo
        time_signal = np.fft.ifft(block)

        # Agregar prefijo cíclico
        time_signal = add_cp(time_signal, CP)

        tx_signal.extend(time_signal)

    return np.array(tx_signal)

In [ ]:
# MODULACIÓN QPSK
symbols_tx = qpsk_modulator(bits_tx)
print("Cantidad de símbolos QPSK:", len(symbols_tx))

In [ ]:
# OFDM
tx_signal = ofdm_transmitter(symbols_tx, N, CP)
print("Longitud de la señal transmitida:", len(tx_signal))

In [ ]:
# Visualización de la señal transmitida
plt.figure(figsize=(12,4))
plt.plot(
    np.real(tx_signal[:400]),
    label="Parte Real (I)"
)
plt.plot(
    np.imag(tx_signal[:400]),
    label="Parte Imaginaria (Q)"
)
plt.title("Señal OFDM en Banda Base")
plt.xlabel("Muestras")
plt.ylabel("Amplitud")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Constelación transmitida
plt.figure(figsize=(6,6))

plt.scatter(
    symbols_tx.real,
    symbols_tx.imag,
    s=4
)
plt.grid(True)
plt.xlabel("In-Phase (I)")
plt.ylabel("Quadrature (Q)")
plt.title("Constelación QPSK Transmitida")
plt.axis("equal")
plt.show()

In [ ]:
# CANAL
def ideal_channel(tx_signal):
    """
    Canal ideal. No modifica la señal transmitida.
    """
    return tx_signal.copy()

In [ ]:
# RECEPTOR OFDM
def ofdm_receiver(rx_signal, N, CP):
    """
    Receptor OFDM.
    Etapas:
        Serie -> Paralelo
        Eliminar CP
        FFT
        Paralelo -> Serie
    """
    symbol_length = N + CP
    n_symbols = len(rx_signal) // symbol_length
    received_symbols = []

    for i in range(n_symbols):
        # Extraer un símbolo OFDM
        start = i * symbol_length
        end = start + symbol_length

        block = rx_signal[start:end]

        # Eliminar prefijo cíclico
        block = remove_cp(block, CP)

        # Volver al dominio de la frecuencia
        freq_signal = np.fft.fft(block)

        received_symbols.extend(freq_signal)

    return np.array(received_symbols)

In [ ]:
# RECEPCIÓN
# Canal
rx_signal = ideal_channel(tx_signal)
# Receptor OFDM
symbols_rx = ofdm_receiver(rx_signal, N, CP)
print("Cantidad de símbolos recibidos:", len(symbols_rx))

In [ ]:
# DEMODULACIÓN QPSK
bits_rx = qpsk_demodulator(symbols_rx)
print("Bits recibidos:", len(bits_rx))

In [ ]:
# BIT ERROR RATE
bit_errors = np.sum(bits_tx != bits_rx)
ber = bit_errors / len(bits_tx)
print("=" * 40)
print("RESULTADOS")
print("=" * 40)
print(f"Bits transmitidos : {len(bits_tx)}")
print(f"Bits erróneos     : {bit_errors}")
print(f"BER               : {ber:.6e}")

In [ ]:
# Constelación recibida
plt.figure(figsize=(6,6))
plt.scatter(
    symbols_rx.real,
    symbols_rx.imag,
    s=4
)
plt.grid(True)
plt.xlabel("In-Phase (I)")
plt.ylabel("Quadrature (Q)")
plt.title("Constelación QPSK Recibida")
plt.axis("equal")
plt.show()

Para poder hacer un poco más de enfasis en el BER, se agrega ruido blanco en la transmición para ver como se comporta la recepcion en un entono más realiasta. 